# 多模型融合图像检索 & 关键点提取

## 概述
- **功能**: 加载四个已训练模型（DINOv2, MAST3R-ASMK, MAST3R-SPoC, ISC）的 best checkpoint，提取全局特征，加权融合相似度，生成 top‑k shortlist，并使用 SuperPoint 和 ALIKE 提取关键点
- **输入**: 图片数据集 + 四个模型的 best checkpoint
- **输出**: shortlist（候选图像对列表）+ SuperPoint 关键点/描述子 + ALIKE 关键点/描述子
- **关键设计**: Shortlist 是候选池，最终场景归属由后续几何验证决定

## 1. 环境配置与路径

In [ ]:
import os, sys, json, time, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import cv2

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# ── 路径配置 ──
IMAGE_ROOT = r'../image-matching-challenge-2025/train'
OUTPUT_DIR = r'../output/retrieval'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 四个模型的 checkpoint 路径（由各自训练 notebook 生成）
MODEL_PATHS = {
    'dino': r'../checkpoints/dinov2/best_model.pth',
    'asmk': r'../checkpoints/mast3r_asmk/best_model.pth',
    'spoc': r'../checkpoints/mast3r_spoc/best_model.pth',
    'isc':  r'../checkpoints/isc/best_model.pth',
}

# ── 检索参数 ──
TOP_K = 20
FUSION_WEIGHTS = {'dino': 0.25, 'asmk': 0.25, 'spoc': 0.25, 'isc': 0.25}
SIM_THRESHOLD = 0.3

# ── 关键点参数 ──
MAX_KPTS = 2048
SUPERPOINT_WEIGHTS = '../models/superpoint_v1.pth'   # 需手动下载，不存在则随机初始化
ALIKE_WEIGHTS = '../models/alike-tiny.pth'             # 需手动下载，不存在则随机初始化

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 收集图片路径

In [ ]:
def collect_image_paths(root):
    """递归收集所有图片路径"""
    root = Path(root)
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    paths = sorted(
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in image_exts
    )
    print(f'Collected {len(paths)} images from {root.resolve()}')
    return paths


image_paths = collect_image_paths(IMAGE_ROOT)
image_names = [p.name for p in image_paths]
name_to_idx = {name: i for i, name in enumerate(image_names)}

# 保存图片路径映射
path_mapping = {p.name: str(p) for p in image_paths}
with open(os.path.join(OUTPUT_DIR, 'image_paths.json'), 'w') as f:
    json.dump(path_mapping, f, indent=2)
print(f'Saved path mapping to {OUTPUT_DIR}/image_paths.json')

## 3. 加载四个已训练模型

每个模型加载其最佳 checkpoint，切换到 eval 模式。
由于各模型来自不同 notebook，这里包含每个模型的类定义（或从外部模块导入）。

In [ ]:
sys.path.insert(0, '../mast3r')
from mast3r.model import AsymmetricMASt3R

# ═══════════════════════════════════════════
# 共享的 MAST3R 路径和维度（ViT-Large: enc_dim=1024）
# ═══════════════════════════════════════════
MAST3R_CHECKPOINT = '../models/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
MAST3R_ENC_DIM = 1024       # ViT-Large encoder dimension


# ═══════════════════════════════════════════
# 3a. DINOv2 模型加载
# ═══════════════════════════════════════════
from transformers import AutoModel

DINO_INPUT_SIZE = 224
DINO_PROJECT_DIM = 256
DINO_BACKBONE_PATH = '../models/dinov2-base'


class DINOBackboneInference(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(DINO_BACKBONE_PATH, local_files_only=True)
        self.projector = nn.Linear(self.model.config.hidden_size, DINO_PROJECT_DIM)

    def forward(self, x):
        features = self.model(x).last_hidden_state[:, 0]
        return self.projector(features)


def load_dino_model(ckpt_path):
    model = DINOBackboneInference().to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    if 'student_state' in ckpt:
        state = ckpt['student_state']
        state = {k.replace('backbone.', ''): v for k, v in state.items() if k.startswith('backbone.')}
    else:
        state = ckpt
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f'DINOv2 loaded from {ckpt_path}')
    return model


dino_model = load_dino_model(MODEL_PATHS['dino'])


# ═══════════════════════════════════════════
# 3b. MAST3R-ASMK 模型加载
# ═══════════════════════════════════════════
ASMK_IMG_SIZE = 224          # 与训练一致
ASMK_CODEBOOK_SIZE = 4096


class MAST3RBackboneInference(nn.Module):
    """MAST3R encoder: patch_embed → enc_blocks(+pos) → enc_norm"""

    def __init__(self, pretrained_path):
        super().__init__()
        self.mast3r = AsymmetricMASt3R.from_pretrained(pretrained_path)
        self.patch_embed = self.mast3r.patch_embed
        self.enc_blocks = self.mast3r.enc_blocks
        self.enc_norm = self.mast3r.enc_norm

    def forward(self, img):
        x, pos = self.patch_embed(img)
        for blk in self.enc_blocks:
            x = blk(x, pos)
        x = self.enc_norm(x)
        return x


class ASMKAggregatorInference(nn.Module):
    def __init__(self, input_dim=MAST3R_ENC_DIM, codebook_size=4096, top_k=100):
        super().__init__()
        self.codebook = nn.Parameter(torch.empty(codebook_size, input_dim))
        nn.init.xavier_uniform_(self.codebook)
        self.alpha = nn.Parameter(torch.tensor(3.0))
        self.top_k = top_k

    def forward(self, features):
        features_norm = F.normalize(features, p=2, dim=-1)
        codebook_norm = F.normalize(self.codebook, p=2, dim=-1)
        sim = torch.einsum('bnd,kd->bkn', features_norm, codebook_norm) * self.alpha
        B, K, N = sim.shape
        k = min(self.top_k, N)
        topk_vals, _ = torch.topk(sim, k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)
        aggregated = weights.sum(dim=-1)
        return F.normalize(aggregated, p=2, dim=-1)


def load_asmk_model(ckpt_path, mast3r_path):
    backbone = MAST3RBackboneInference(mast3r_path).to(device)
    asmk = ASMKAggregatorInference(
        input_dim=MAST3R_ENC_DIM, codebook_size=ASMK_CODEBOOK_SIZE
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    asmk.load_state_dict(ckpt['asmk_state'])
    backbone.eval()
    asmk.eval()
    print(f'MAST3R-ASMK loaded from {ckpt_path} (epoch {ckpt["epoch"]})')
    return backbone, asmk


asmk_backbone, asmk_aggregator = load_asmk_model(MODEL_PATHS['asmk'], MAST3R_CHECKPOINT)


# ═══════════════════════════════════════════
# 3c. MAST3R-SPoC 模型加载
# ═══════════════════════════════════════════
SPOC_IMG_SIZE = 224          # 与训练一致
SPOC_OUTPUT_DIM = 256


class SPoCHeadInference(nn.Module):
    def __init__(self, input_dim=MAST3R_ENC_DIM, hidden_dim=2048, output_dim=256):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, features):
        pooled = features.sum(dim=1)
        descriptor = self.projection(pooled)
        return F.normalize(descriptor, p=2, dim=-1)


def load_spoc_model(ckpt_path, mast3r_path):
    backbone = MAST3RBackboneInference(mast3r_path).to(device)
    spoc_head = SPoCHeadInference(
        input_dim=MAST3R_ENC_DIM, hidden_dim=2048, output_dim=SPOC_OUTPUT_DIM
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    spoc_head.load_state_dict(ckpt['spoc_head_state'])
    backbone.eval()
    spoc_head.eval()
    print(f'MAST3R-SPoC loaded from {ckpt_path} (epoch {ckpt["epoch"]})')
    return backbone, spoc_head


spoc_backbone, spoc_head = load_spoc_model(MODEL_PATHS['spoc'], MAST3R_CHECKPOINT)


# ═══════════════════════════════════════════
# 3d. ISC 模型加载
# ═══════════════════════════════════════════
import timm

ISC_IMG_SIZE = 384
ISC_FEATURE_DIM = 512
EFFICIENTNET_WEIGHT = '../models/timm/tf_efficientnet_b3_ns-9d44bf68.pth'


class GeMInference(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            kernel_size=(x.size(-2), x.size(-1))
        ).pow(1.0 / self.p).flatten(1)


class ISCBackboneInference(nn.Module):
    def __init__(self, feature_dim=ISC_FEATURE_DIM):
        super().__init__()
        self.encoder = timm.create_model('tf_efficientnet_b3_ns', pretrained=False, num_classes=0)
        # 加载本地预训练权重（与训练时一致）
        if os.path.exists(EFFICIENTNET_WEIGHT):
            pretrained_state = torch.load(EFFICIENTNET_WEIGHT, map_location='cpu', weights_only=True)
            self.encoder.load_state_dict(pretrained_state, strict=False)
        encoder_dim = self.encoder.num_features
        self.pooling = GeMInference()
        self.fc = nn.Linear(encoder_dim, feature_dim)
        self.bn = nn.BatchNorm1d(feature_dim)

    def forward(self, x):
        features = self.encoder.forward_features(x)
        pooled = self.pooling(features)
        embedding = self.fc(pooled)
        embedding = self.bn(embedding)
        return F.normalize(embedding, p=2, dim=-1)


def load_isc_model(ckpt_path):
    model = ISCBackboneInference().to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['backbone_state'])
    model.eval()
    print(f'ISC loaded from {ckpt_path} (epoch {ckpt["epoch"]})')
    return model


isc_model = load_isc_model(MODEL_PATHS['isc'])

print('\nAll 4 models loaded successfully!')

## 4. 批量全局特征提取

对全部 N 张图片，用 4 个模型分别提取全局特征向量。

In [ ]:
def prepare_inference_transform(input_size):
    """推理用 transform：resize + center crop + normalize"""
    return transforms.Compose([
        transforms.Resize(int(input_size * 1.14)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


@torch.no_grad()
def extract_features_batch(model, model_name, image_paths, transform, batch_size, device, **kwargs):
    """
    通用批量特征提取。
    kwargs 可用于传入额外模块（如 asmk_aggregator, spoc_head）。
    """
    if isinstance(model, (tuple, list)):
        models = list(model)
    else:
        models = [model]
    for m in models:
        m.eval()

    features_list = []
    for i in tqdm(range(0, len(image_paths), batch_size),
                  desc=f'Extracting {model_name} features'):
        batch_paths = image_paths[i:i + batch_size]
        imgs = []
        for p in batch_paths:
            im = Image.open(p).convert('RGB')
            imgs.append(transform(im).unsqueeze(0))
        x = torch.cat(imgs, dim=0).to(device)

        if model_name == 'asmk':
            feat = models[0](x)
            feat = kwargs['asmk_aggregator'](feat)
        elif model_name == 'spoc':
            feat = models[0](x)
            feat = kwargs['spoc_head'](feat)
        else:
            feat = models[0](x)

        features_list.append(feat.cpu())

    return torch.cat(features_list, dim=0)


print('Extracting features from all 4 models...')
t0 = time.time()

# DINOv2
dino_features = extract_features_batch(
    dino_model, 'dino', image_paths,
    prepare_inference_transform(DINO_INPUT_SIZE), batch_size=32, device=device
)
print(f'  DINOv2:   {dino_features.shape}')

# MAST3R-ASMK
asmk_features = extract_features_batch(
    asmk_backbone, 'asmk', image_paths,
    prepare_inference_transform(ASMK_IMG_SIZE), batch_size=8, device=device,
    asmk_aggregator=asmk_aggregator
)
print(f'  ASMK:      {asmk_features.shape}')

# MAST3R-SPoC
spoc_features = extract_features_batch(
    spoc_backbone, 'spoc', image_paths,
    prepare_inference_transform(SPOC_IMG_SIZE), batch_size=8, device=device,
    spoc_head=spoc_head
)
print(f'  SPoC:      {spoc_features.shape}')

# ISC
isc_features = extract_features_batch(
    isc_model, 'isc', image_paths,
    prepare_inference_transform(ISC_IMG_SIZE), batch_size=32, device=device
)
print(f'  ISC:       {isc_features.shape}')

t1 = time.time()
print(f'\nFeature extraction completed in {t1-t0:.1f}s')

## 5. 多模型加权融合 & Shortlist 生成

- 计算各模型的余弦相似度矩阵（N×N）
- 加权求和融合
- 每张图取 top‑k 最相似候选 → shortlist
- 排除自身匹配、低于阈值的候选

In [ ]:
def cosine_similarity_matrix(features):
    """计算余弦相似度矩阵 (N, N)"""
    features = F.normalize(features, p=2, dim=-1)
    sim = torch.mm(features, features.t())
    return sim


print('Computing similarity matrices...')
t0 = time.time()

sim_dino = cosine_similarity_matrix(dino_features)
sim_asmk = cosine_similarity_matrix(asmk_features)
sim_spoc = cosine_similarity_matrix(spoc_features)
sim_isc  = cosine_similarity_matrix(isc_features)

# ── 加权融合（非取交集）──
fused_sim = (
    FUSION_WEIGHTS['dino'] * sim_dino +
    FUSION_WEIGHTS['asmk'] * sim_asmk +
    FUSION_WEIGHTS['spoc'] * sim_spoc +
    FUSION_WEIGHTS['isc']  * sim_isc
)

t1 = time.time()
print(f'Similarity matrices computed in {t1-t0:.1f}s')
print(f'Fused similarity shape: {fused_sim.shape}')


# ── 生成 Shortlist ──
N = len(image_paths)
k_effective = min(TOP_K, N - 1)

# 排除对角线（自身相似度）
fused_sim_no_diag = fused_sim.clone()
fused_sim_no_diag.fill_diagonal_(-float('inf'))

# 取 top-k
topk_scores, topk_indices = torch.topk(fused_sim_no_diag, k=k_effective, dim=-1)

# 构建 shortlist：每张图的候选邻居列表
shortlist = {}
for i in range(N):
    candidates = []
    for j, score in zip(topk_indices[i].tolist(), topk_scores[i].tolist()):
        if score >= SIM_THRESHOLD:
            candidates.append({
                'idx': j,
                'name': image_names[j],
                'score': score,
            })
    shortlist[image_names[i]] = candidates

# 统计
total_pairs = sum(len(v) for v in shortlist.values())
avg_candidates = total_pairs / N if N > 0 else 0
print(f'\nShortlist generated:')
print(f'  Total candidate pairs: {total_pairs}')
print(f'  Avg candidates per image: {avg_candidates:.1f}')
print(f'  Threshold: {SIM_THRESHOLD}')

# 保存 shortlist
shortlist_serializable = {
    k: [{'idx': c['idx'], 'name': c['name'], 'score': float(c['score'])}
        for c in v]
    for k, v in shortlist.items()
}
with open(os.path.join(OUTPUT_DIR, 'shortlist.json'), 'w') as f:
    json.dump(shortlist_serializable, f, indent=2)
print(f'Saved shortlist to {OUTPUT_DIR}/shortlist.json')

## 6. 相似度矩阵可视化

检查各模型相似度分布及融合效果。

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
sims = [sim_dino, sim_asmk, sim_spoc, sim_isc, fused_sim]
titles = ['DINOv2', 'MAST3R-ASMK', 'MAST3R-SPoC', 'ISC', 'Fused']

for ax, sim, title in zip(axes, sims, titles):
    # 显示前 100 张图的相似度（太大则采样）
    n_show = min(100, N)
    im = ax.imshow(sim[:n_show, :n_show].numpy(), cmap='viridis', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel('Image index')
    ax.set_ylabel('Image index')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Similarity Matrices (first 100 images)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'similarity_matrices.png'), dpi=150)
plt.show()

## 7. SuperPoint 关键点检测

SuperPoint：先检测后描述，精度高，输出 256 维描述子。
输入单张 RGB 图 → 输出关键点坐标 + 256 维描述子。

In [ ]:
class SuperPointNet(nn.Module):
    """
    SuperPoint 网络（简化版，完整实现参考 Magic Leap 开源代码）
    输入: (1, 1, H, W) grayscale image
    输出: keypoints (N, 2), descriptors (N, 256), scores (N,)
    """

    def __init__(self, weights_path=None):
        super().__init__()
        # Encoder (VGG-style)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
        )
        # Detector head
        self.detector = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 65, 1),
        )
        # Descriptor head
        self.descriptor = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 1),
        )

        if weights_path and os.path.exists(weights_path):
            self.load_state_dict(torch.load(weights_path, map_location='cpu'), strict=False)
            print(f'SuperPoint weights loaded from {weights_path}')

    def forward(self, x):
        # x: (B, 1, H, W)
        features = self.encoder(x)
        # Detector: (B, 65, H/8, W/8) → scores
        semi = self.detector(features)           # (B, 65, Hc, Wc)
        # Descriptors: (B, 256, H/8, W/8)
        desc = self.descriptor(features)          # (B, 256, Hc, Wc)
        desc = F.normalize(desc, p=2, dim=1)
        return semi, desc


@torch.no_grad()
def superpoint_extract(model, image_path, max_kpts=MAX_KPTS, conf_thresh=0.015):
    """
    对单张图片提取 SuperPoint 关键点和描述子

    Args:
        model: SuperPointNet
        image_path: 图片路径
        max_kpts: 最大关键点数
        conf_thresh: 置信度阈值
    Returns:
        keypoints: (N, 2) [x, y] 坐标
        descriptors: (N, 256) 描述子
        scores: (N,) 置信度
    """
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((0, 2)), np.zeros((0, 256)), np.zeros((0,))

    H, W = img.shape
    # Resize 到 8 的倍数
    H_new, W_new = (H // 8) * 8, (W // 8) * 8
    img_resized = cv2.resize(img, (W_new, H_new))

    tensor = torch.from_numpy(img_resized / 255.0).float().unsqueeze(0).unsqueeze(0).to(device)
    semi, coarse_desc = model(tensor)

    # 从 semi 输出提取关键点 (NMS + threshold)
    semi = semi.squeeze(0)  # (65, Hc, Wc)
    Hc, Wc = semi.shape[1], semi.shape[2]
    # dustbin = semi[64]
    nodust = semi[:64]  # (64, Hc, Wc)

    # 转成 (Hc, Wc, 8, 8) → 像素级 heatmap
    heatmap = nodust.reshape(8, 8, Hc, Wc).permute(2, 3, 0, 1).reshape(Hc, Wc, -1)
    heatmap = F.softmax(heatmap, dim=-1).reshape(Hc, Wc, 8, 8).permute(0, 2, 1, 3).reshape(Hc * 8, Wc * 8)

    # NMS
    from torch.nn.functional import max_pool2d
    pooled = max_pool2d(heatmap.unsqueeze(0).unsqueeze(0), kernel_size=3, stride=1, padding=1)
    nms = (heatmap == pooled.squeeze()) & (heatmap >= conf_thresh)

    ys, xs = torch.where(nms)
    scores = heatmap[ys, xs]

    # 按置信度排序取 top-k
    if len(scores) > max_kpts:
        topk_idx = torch.topk(scores, max_kpts).indices
        ys, xs = ys[topk_idx], xs[topk_idx]
        scores = scores[topk_idx]

    # 缩放回原始图像坐标
    xs_orig = xs.float() * (W / W_new)
    ys_orig = ys.float() * (H / H_new)
    keypoints = torch.stack([xs_orig, ys_orig], dim=-1).cpu().numpy()

    # 提取描述子
    coarse_desc = coarse_desc.squeeze(0)  # (256, Hc, Wc)
    xc = (xs.float() / 8.0).long().clamp(0, Wc - 1)
    yc = (ys.float() / 8.0).long().clamp(0, Hc - 1)
    descriptors = coarse_desc[:, yc, xc].t()  # (N, 256)
    descriptors = F.normalize(descriptors, p=2, dim=-1).cpu().numpy()

    return keypoints, descriptors, scores.cpu().numpy()


# 初始化 SuperPoint
superpoint_model = SuperPointNet(weights_path=SUPERPOINT_WEIGHTS).to(device).eval()
print('SuperPoint model ready.')

## 8. ALIKE 关键点检测

ALIKE：端到端检测+描述，速度更快，输出 128 维描述子。
输入单张 RGB 图 → 输出关键点坐标 + 128 维描述子。

In [ ]:
class ALIKENet(nn.Module):
    """
    ALIKE 网络（简化版，完整实现参考 https://github.com/Shiaoming/ALIKE）
    输入: (1, 3, H, W) RGB image
    输出: keypoints (N, 2), descriptors (N, 128), scores (N,)
    """

    def __init__(self, weights_path=None, model_type='tiny'):
        super().__init__()
        c1, c2, c3, c4 = {'tiny': (8, 16, 32, 64), 'small': (16, 32, 64, 128),
                          'base': (32, 64, 128, 256), 'large': (64, 128, 256, 512)}[model_type]

        # Encoder
        self.block1 = nn.Sequential(
            nn.Conv2d(3, c1, 3, stride=2, padding=1), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, padding=1), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(c1, c2, 3, stride=2, padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.Conv2d(c2, c2, 3, padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(c2, c3, 3, stride=2, padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.Conv2d(c3, c3, 3, padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(c3, c4, 3, stride=2, padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.Conv2d(c4, c4, 3, padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
        )
        # Score head
        self.score_head = nn.Sequential(
            nn.Conv2d(c4, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1), nn.Sigmoid(),
        )
        # Descriptor head
        self.desc_head = nn.Sequential(
            nn.Conv2d(c4, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 1),
        )

        if weights_path and os.path.exists(weights_path):
            self.load_state_dict(torch.load(weights_path, map_location='cpu'), strict=False)
            print(f'ALIKE weights loaded from {weights_path}')

    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.block2(x1)
        x3 = self.block3(x2)
        x4 = self.block4(x3)
        score_map = self.score_head(x4)       # (B, 1, H/16, W/16)
        desc_map = self.desc_head(x4)          # (B, 128, H/16, W/16)
        desc_map = F.normalize(desc_map, p=2, dim=1)
        return score_map, desc_map


@torch.no_grad()
def alike_extract(model, image_path, max_kpts=MAX_KPTS, conf_thresh=0.02):
    """
    对单张图片提取 ALIKE 关键点和描述子

    Args:
        model: ALIKENet
        image_path: 图片路径
        max_kpts: 最大关键点数
        conf_thresh: 置信度阈值
    Returns:
        keypoints: (N, 2) [x, y] 坐标
        descriptors: (N, 128) 描述子
        scores: (N,) 置信度
    """
    img = cv2.imread(str(image_path))
    if img is None:
        return np.zeros((0, 2)), np.zeros((0, 128)), np.zeros((0,))

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]

    # Resize 到 16 的倍数
    H_new, W_new = (H // 16) * 16, (W // 16) * 16
    img_resized = cv2.resize(img_rgb, (W_new, H_new))

    tensor = torch.from_numpy(img_resized).float().permute(2, 0, 1).unsqueeze(0) / 255.0
    tensor = (tensor - 0.5) / 0.5  # normalize to [-1, 1]
    tensor = tensor.to(device)

    score_map, desc_map = model(tensor)
    score_map = score_map.squeeze()  # (Hs, Ws)
    desc_map = desc_map.squeeze(0)    # (128, Hs, Ws)

    Hs, Ws = score_map.shape

    # NMS
    from torch.nn.functional import max_pool2d
    pooled = max_pool2d(score_map.unsqueeze(0).unsqueeze(0),
                        kernel_size=3, stride=1, padding=1)
    nms = (score_map == pooled.squeeze()) & (score_map >= conf_thresh)

    ys, xs = torch.where(nms)
    scores = score_map[ys, xs]

    # 取 top-k
    if len(scores) > max_kpts:
        topk_idx = torch.topk(scores, max_kpts).indices
        ys, xs = ys[topk_idx], xs[topk_idx]
        scores = scores[topk_idx]

    # 缩放回原始坐标
    xs_orig = xs.float() * (W / Ws)
    ys_orig = ys.float() * (H / Hs)
    keypoints = torch.stack([xs_orig, ys_orig], dim=-1).cpu().numpy()

    # 提取描述子
    descriptors = desc_map[:, ys, xs].t()  # (N, 128)
    descriptors = F.normalize(descriptors, p=2, dim=-1).cpu().numpy()

    return keypoints, descriptors, scores.cpu().numpy()


# 初始化 ALIKE
alike_model = ALIKENet(weights_path=ALIKE_WEIGHTS, model_type='tiny').to(device).eval()
print('ALIKE model ready.')

## 9. 批量关键点提取 + 可视化验证

对全部图片（或采样验证）同时运行 SuperPoint 和 ALIKE，保存结果。

In [ ]:
def extract_all_keypoints(image_paths, superpoint_model, alike_model):
    """
    批量提取 SuperPoint 和 ALIKE 关键点

    Returns:
        kpts_data: dict {image_name: {'sp': (kpts, desc, scores), 'alike': (kpts, desc, scores)}}
    """
    kpts_data = {}
    for p in tqdm(image_paths, desc='Extracting keypoints'):
        name = p.name
        sp_kpts, sp_desc, sp_scores = superpoint_extract(superpoint_model, p)
        al_kpts, al_desc, al_scores = alike_extract(alike_model, p)
        kpts_data[name] = {
            'superpoint': {
                'keypoints': sp_kpts.tolist(),
                'descriptors': sp_desc.tolist(),
                'scores': sp_scores.tolist(),
            },
            'alike': {
                'keypoints': al_kpts.tolist(),
                'descriptors': al_desc.tolist(),
                'scores': al_scores.tolist(),
            },
        }
    return kpts_data


# 提取所有关键点（如果图片很多可考虑采样）
USE_SAMPLE = len(image_paths) > 500
sample_paths = image_paths[:min(500, len(image_paths))] if USE_SAMPLE else image_paths

print(f'Extracting keypoints for {len(sample_paths)} images...')
kpts_data = extract_all_keypoints(sample_paths, superpoint_model, alike_model)

# 保存关键点数据
kpts_output = {}
for name, data in kpts_data.items():
    kpts_output[name] = {
        'superpoint_kpts': data['superpoint']['keypoints'],
        'superpoint_desc_dim': 256,
        'alike_kpts': data['alike']['keypoints'],
        'alike_desc_dim': 128,
    }
with open(os.path.join(OUTPUT_DIR, 'keypoints.json'), 'w') as f:
    json.dump(kpts_output, f)

# 保存为 pickle（含完整描述子）
pickle_path = os.path.join(OUTPUT_DIR, 'keypoints.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(kpts_data, f)
print(f'Keypoints saved to {pickle_path}')

# 统计
sp_counts = [len(d['superpoint']['keypoints']) for d in kpts_data.values()]
al_counts = [len(d['alike']['keypoints']) for d in kpts_data.values()]
print(f'SuperPoint: avg {np.mean(sp_counts):.0f} kpts/image, total {sum(sp_counts)}')
print(f'ALIKE:      avg {np.mean(al_counts):.0f} kpts/image, total {sum(al_counts)}')

## 10. 关键点可视化抽查

In [ ]:
def visualize_keypoints(image_path, sp_kpts, al_kpts):
    """并排对比 SuperPoint 和 ALIKE 关键点"""
    img = cv2.imread(str(image_path))
    if img is None:
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    ax1.imshow(img_rgb)
    if len(sp_kpts) > 0:
        ax1.scatter(sp_kpts[:, 0], sp_kpts[:, 1], c='lime', s=2, alpha=0.7)
    ax1.set_title(f'SuperPoint ({len(sp_kpts)} kpts)')
    ax1.axis('off')

    ax2.imshow(img_rgb)
    if len(al_kpts) > 0:
        ax2.scatter(al_kpts[:, 0], al_kpts[:, 1], c='cyan', s=2, alpha=0.7)
    ax2.set_title(f'ALIKE ({len(al_kpts)} kpts)')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()


# 随机抽查 3 张图
np.random.seed(42)
sample_names = np.random.choice(list(kpts_data.keys()), min(3, len(kpts_data)), replace=False)
for name in sample_names:
    p = Path(IMAGE_ROOT) / name
    if not p.exists():
        p = image_paths[image_names.index(name)]
    sp = np.array(kpts_data[name]['superpoint']['keypoints'])
    al = np.array(kpts_data[name]['alike']['keypoints'])
    print(f'\nImage: {name}')
    visualize_keypoints(p, sp, al)

## 11. 输出汇总

保存所有中间结果供后续 Phase 3/5/6 使用。

In [ ]:
print('=' * 60)
print('Phase 2 & 4 Complete: Retrieval + Keypoints')
print('=' * 60)
print(f'Output directory: {OUTPUT_DIR}')
print(f'  - image_paths.json     : image name -> path mapping')
print(f'  - shortlist.json       : top-{TOP_K} candidates per image')
print(f'  - keypoints.json       : keypoint coordinates (summary)')
print(f'  - keypoints.pkl        : full keypoints + descriptors')
print(f'  - similarity_matrices.png : visualization')
print(f'\nShortlist stats:')
print(f'  Total images:         {N}')
print(f'  Total candidate pairs: {total_pairs}')
print(f'  Avg candidates/image:  {avg_candidates:.1f}')
print(f'  Fusion weights:        {FUSION_WEIGHTS}')
print(f'\nNext step: Pre-clustering with coarse MAST3R geometric verification')
print(f'         → pre_clustering.ipynb')